# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from groq import Groq
import gradio as gr

In [4]:
# Initialization

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')
groq_api_model = os.getenv('GROQ_API_MODEL')

groq = Groq(api_key=groq_api_key)

if groq_api_key:
    print(f"groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("groq API Key not set")

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


groq API Key exists and begins gsk_23vF


In [6]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [7]:
def chat(message, history):

    print(f"system_message in chat is {system_message}")
    print(f"message is chat is {message}")

    history = [{"role":h["role"], "content":h["content"]} for h in history]
    print(f"history is chat is {history}")
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(f"full message sent to LLM is {messages}")
   
    response = groq.chat.completions.create(model=groq_api_model, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch(debug=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [8]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [9]:
print(get_ticket_price("paris"))
print(get_ticket_price("berlin"))
print(get_ticket_price("melbourne"))

Tool called for city paris
The price of a ticket to paris is $899
Tool called for city berlin
The price of a ticket to berlin is $499
Tool called for city melbourne
The price of a ticket to melbourne is Unknown ticket price


In [10]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [11]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [12]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [24]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    print(f"--------------------------------handle_tool_call start-------------------------------------------")
    print(f"message is {message}")
    tool_call = message.tool_calls[0]
    print(f"tool_call is {tool_call}")
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        print(f"arguments are {arguments}")
        city = arguments.get('destination_city')
        print(f"city is {city}")
        price_details = get_ticket_price(city)
        print(f"price_details is {price_details}")
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    print(f"--------------------------------handle_tool_call end---------------------------------------")
    return response

In [25]:
def chat(message, history):
    print(f"--------------------------------chat start---------------------------------------------------------------------------------------------------------")
    print(f"system_message in chat is {system_message}")
    print(f"message is chat is {message}")
    print(f"history chat is {history}")
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    print(f"updated history chat(only role and contnet attribute) is {history}")
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(f"formatted messages(System message + history(both user and system message) + current message from user) are which being sent to model \n {messages}")
    response = groq.chat.completions.create(model=groq_api_model, messages=messages, tools=tools)

    print(f"response1 is \n {response}")
    print(f"response.choices[0].finish_reason>>  {response.choices[0].finish_reason}")
    
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"reponse message to pass into handle tool call is>>> {message}")
        response = handle_tool_call(message)
        print(f"response2 from handle tool call is {response}")
        messages.append(message)
        
        for my_message in messages:
        #   this is one liner for loop just printing all messages
          print(f"my message is: {my_message}")

        response = groq.chat.completions.create(model=groq_api_model, messages=messages)
        print(f"response3 is {response}")

    else:
        print(f"-----TOOL CALL DID NOT EXECUTE------")
    print(f"--------------------------------chat end---------------------------------------------------------------------------------------------------------")
    return response.choices[0].message.content

   <!-- Response 1 message when we send Hi message
ChatCompletion(id='chatcmpl-48584356-cef0-4cb7-ae5a-482b4286787e',
    choices=[
        Choice(
            finish_reason='tool_calls',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                role='assistant',
                executed_tools=None,
                function_call=None,
                reasoning=(
                    'User wants to travel to London. Likely wants price. '
                    'We can ask? But instructions: short answer, but we can provide '
                    'price via function. We need to get price. Use get_ticket_price '
                    'function with destination_city "London".'
                ),
                tool_calls=[
                    ChatCompletionMessageToolCall(
                        id='fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93',
                        function=Function(
                            arguments='{"destination_city":"London"}',
                            name='get_ticket_price'
                        ),
                        type='function'
                    )
                ]
            )
        )
    ],
    created=1791258218,
    model='openai/gpt-oss-120b',
    object='chat.completion',
    system_fingerprint='fp_854fa9be4c',
    usage=CompletionUsage(
        completion_tokens=77,
        prompt_tokens=212,
        total_tokens=289,
        completion_time=0.159559244,
        prompt_time=0.009620063,
        queue_time=0.275270685,
        total_time=0.169179307,
        completion_tokens_details={
            'reasoning_tokens': 48
        }
    ),
    usage_breakdown=None,
    x_groq={
        'id': 'req_01m47myk9me0c8bfkve8f8wf2t',
        'seed': 1238571457
    },
    service_tier='on_demand'
) -->


reposnse 1 merssage when we send message that I would like to travel to london 



<!-- Response 1 message when we send Hi message
ChatCompletion(
    id='chatcmpl-63242d1c-8d4a-4f68-b8e2-ed3dc9ec564a',
    choices=[
        Choice(
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content='Hello! How can I assist you with your travel plans today?',
                role='assistant',
                executed_tools=None,
                function_call=None,
                reasoning=(
                    'We need to respond short, courteous, max 1 sentence. '
                    'User just says "Hi". Respond accordingly.'
                ),
                tool_calls=None
            )
        )
    ],
    created=1791181961,
    model='openai/gpt-oss-120b',
    object='chat.completion',
    system_fingerprint='fp_e1a78f200e',
    usage=CompletionUsage(
        completion_tokens=45,
        prompt_tokens=186,
        total_tokens=231,
        completion_time=0.093288901,
        prompt_time=0.039105966,
        queue_time=0.327463799,
        total_time=0.132394867,
        completion_tokens_details={'reasoning_tokens': 23}
    ),
    usage_breakdown=None,
    x_groq={
        'id': 'req_01m45c7dvze3ktv60x8ww9whvg',
        'seed': 1353787455
    },
    service_tier='on_demand'
)
 -->


<!-- reposnse 1 merssage when we send message that I would like to travel to london 

ChatCompletion(
    id='chatcmpl-48584356-cef0-4cb7-ae5a-482b4286787e',
    choices=[
        Choice(
            finish_reason='tool_calls',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                role='assistant',
                executed_tools=None,
                function_call=None,
                reasoning=(
                    'User wants to travel to London. Likely wants price. '
                    'We can ask? But instructions: short answer, but we can provide '
                    'price via function. We need to get price. Use get_ticket_price '
                    'function with destination_city "London".'
                ),
                tool_calls=[
                    ChatCompletionMessageToolCall(
                        id='fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93',
                        function=Function(
                            arguments='{"destination_city":"London"}',
                            name='get_ticket_price'
                        ),
                        type='function'
                    )
                ]
            )
        )
    ],
    created=1791258218,
    model='openai/gpt-oss-120b',
    object='chat.completion',
    system_fingerprint='fp_854fa9be4c',
    usage=CompletionUsage(
        completion_tokens=77,
        prompt_tokens=212,
        total_tokens=289,
        completion_time=0.159559244,
        prompt_time=0.009620063,
        queue_time=0.275270685,
        total_time=0.169179307,
        completion_tokens_details={'reasoning_tokens': 48}
    ),
    usage_breakdown=None,
    x_groq={
        'id': 'req_01m47myk9me0c8bfkve8f8wf2t',
        'seed': 1238571457
    },
    service_tier='on_demand'
) -->


<!-- reponse message to pass into handle tool call is>>> 

ChatCompletionMessage(
    content=None,
    role='assistant',
    executed_tools=None,
    function_call=None,
    reasoning=(
        'User wants to travel to London. Likely wants price. '
        'We can ask? But instructions: short answer, but we can provide price '
        'via function. We need to get price. Use get_ticket_price function '
        'with destination_city "London".'
    ),
    tool_calls=[
        ChatCompletionMessageToolCall(
            id='fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93',
            function=Function(
                arguments='{"destination_city":"London"}',
                name='get_ticket_price'
            ),
            type='function'
        )
    ]
) -->

<!-- 
response2 from handle tool call:
{
  "role": "tool",
  "content": "The price of a ticket to London is $799",
  "tool_call_id": "fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93"
} -->
<!--input message for class call  -->

<!-- input message for final call is 
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'Hi'}
{'role': 'assistant', 'content': 'Hello! How can I assist you today?'}
{'role': 'user', 'content': 'I would like to travel to london'}
ChatCompletionMessage(content=None, role='assistant', executed_tools=None, function_call=None, reasoning='User wants to travel to London. Likely wants price. We can ask? But instructions: short answer, but we can provide price via function. We need to get price. Use get_ticket_price function with destination_city "London".', tool_calls=[ChatCompletionMessageToolCall(id='fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')])

 -->

<!-- final response3 is

{
  "id": "chatcmpl-5efe6725-1fb7-4d94-85f4-12a5ce3bf709",
  "object": "chat.completion",
  "model": "openai/gpt-oss-120b",
  "created": 1791258218,
  "system_fingerprint": "fp_60e4b492db",
  "choices": [
    {
      "index": 0,
      "finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "The current ticket price to London is $1,150.",
        "reasoning": "We need response with price. After function result, we respond.",
        "executed_tools": null,
        "function_call": null,
        "tool_calls": null
      },
      "logprobs": null
    }
  ],
  "usage": {
    "prompt_tokens": 210,
    "completion_tokens": 35,
    "total_tokens": 245,
    "completion_time": 0.07437133,
    "prompt_time": 0.010283456,
    "queue_time": 0.263090433,
    "total_time": 0.084654786,
    "completion_tokens_details": {
      "reasoning_tokens": 14
    }
  },
  "usage_breakdown": null,
  "x_groq": {
    "id": "req_01m47mykvme1rvcxjzs1x8g94h",
    "seed": 1101646010
  },
  "service_tier": "on_demand"
} -->


<!-- Response 1 message when we send Hi message
ChatCompletion(
    id='chatcmpl-63242d1c-8d4a-4f68-b8e2-ed3dc9ec564a',
    choices=[
        Choice(
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content='Hello! How can I assist you with your travel plans today?',
                role='assistant',
                executed_tools=None,
                function_call=None,
                reasoning=(
                    'We need to respond short, courteous, max 1 sentence. '
                    'User just says "Hi". Respond accordingly.'
                ),
                tool_calls=None
            )
        )
    ],
    created=1791181961,
    model='openai/gpt-oss-120b',
    object='chat.completion',
    system_fingerprint='fp_e1a78f200e',
    usage=CompletionUsage(
        completion_tokens=45,
        prompt_tokens=186,
        total_tokens=231,
        completion_time=0.093288901,
        prompt_time=0.039105966,
        queue_time=0.327463799,
        total_time=0.132394867,
        completion_tokens_details={'reasoning_tokens': 23}
    ),
    usage_breakdown=None,
    x_groq={
        'id': 'req_01m45c7dvze3ktv60x8ww9whvg',
        'seed': 1353787455
    },
    service_tier='on_demand'
)
 -->


<!-- reposnse 1 merssage when we send message that I would like to travel to london 

ChatCompletion(
    id='chatcmpl-48584356-cef0-4cb7-ae5a-482b4286787e',
    choices=[
        Choice(
            finish_reason='tool_calls',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                role='assistant',
                executed_tools=None,
                function_call=None,
                reasoning=(
                    'User wants to travel to London. Likely wants price. '
                    'We can ask? But instructions: short answer, but we can provide '
                    'price via function. We need to get price. Use get_ticket_price '
                    'function with destination_city "London".'
                ),
                tool_calls=[
                    ChatCompletionMessageToolCall(
                        id='fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93',
                        function=Function(
                            arguments='{"destination_city":"London"}',
                            name='get_ticket_price'
                        ),
                        type='function'
                    )
                ]
            )
        )
    ],
    created=1791258218,
    model='openai/gpt-oss-120b',
    object='chat.completion',
    system_fingerprint='fp_854fa9be4c',
    usage=CompletionUsage(
        completion_tokens=77,
        prompt_tokens=212,
        total_tokens=289,
        completion_time=0.159559244,
        prompt_time=0.009620063,
        queue_time=0.275270685,
        total_time=0.169179307,
        completion_tokens_details={'reasoning_tokens': 48}
    ),
    usage_breakdown=None,
    x_groq={
        'id': 'req_01m47myk9me0c8bfkve8f8wf2t',
        'seed': 1238571457
    },
    service_tier='on_demand'
) -->


<!-- reponse message to pass into handle tool call is>>> 

ChatCompletionMessage(
    content=None,
    role='assistant',
    executed_tools=None,
    function_call=None,
    reasoning=(
        'User wants to travel to London. Likely wants price. '
        'We can ask? But instructions: short answer, but we can provide price '
        'via function. We need to get price. Use get_ticket_price function '
        'with destination_city "London".'
    ),
    tool_calls=[
        ChatCompletionMessageToolCall(
            id='fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93',
            function=Function(
                arguments='{"destination_city":"London"}',
                name='get_ticket_price'
            ),
            type='function'
        )
    ]
) -->

<!-- 
response2 from handle tool call:
{
  "role": "tool",
  "content": "The price of a ticket to London is $799",
  "tool_call_id": "fc_1651bc16-69dd-49e8-9273-ef4b7efcbe93"
} -->
<!--input message for class call  -->

<!-- final response3 is

{
  "id": "chatcmpl-5efe6725-1fb7-4d94-85f4-12a5ce3bf709",
  "object": "chat.completion",
  "model": "openai/gpt-oss-120b",
  "created": 1791258218,
  "system_fingerprint": "fp_60e4b492db",
  "choices": [
    {
      "index": 0,
      "finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "The current ticket price to London is $1,150.",
        "reasoning": "We need response with price. After function result, we respond.",
        "executed_tools": null,
        "function_call": null,
        "tool_calls": null
      },
      "logprobs": null
    }
  ],
  "usage": {
    "prompt_tokens": 210,
    "completion_tokens": 35,
    "total_tokens": 245,
    "completion_time": 0.07437133,
    "prompt_time": 0.010283456,
    "queue_time": 0.263090433,
    "total_time": 0.084654786,
    "completion_tokens_details": {
      "reasoning_tokens": 14
    }
  },
  "usage_breakdown": null,
  "x_groq": {
    "id": "req_01m47mykvme1rvcxjzs1x8g94h",
    "seed": 1101646010
  },
  "service_tier": "on_demand"
} -->






In [26]:
#  we hae sublte problem that if we provide two city name it gets confused
gr.ChatInterface(fn=chat, type="messages").launch(debug=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


--------------------------------chat start---------------------------------------------------------------------------------------------------------
system_message in chat is 
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

message is chat is Hi
history chat is []
updated history chat(only role and contnet attribute) is []
formatted messages(System message + history(both user and system message) + current message from user) are which being sent to model 
 [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'Hi'}]
response1 is 
 ChatCompletion(id='chatcmpl-950ec1a2-15a2-4f4a-8788-4f5428b1a4c9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompleti

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [46]:
def handle_tool_calls(message):
    responses = []
    print(f"Message in handle_tool_call is >>>>>>>>>> {message}")
    for tool_call in message.tool_calls:
        print(f"And the tool call is >>>>>>>>>>>>>>>>>>>> {tool_call}")
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [47]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(f"message array is>>>>>>>>>>>>>>>>>>>>>>>>> {messages}")
    response = groq.chat.completions.create(model=groq_api_model, messages=messages, tools=tools)
    print(f"reponse of message array is >>>>>>>>>>>>> {response}")
    print(f"it has finish reason >>>>>>>>>>>>>>>>>>>> {response.choices[0].finish_reason}")

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        # here we have not passed tool. so in one message we can provide only one city name.
        # even though it is while loop as tool is not provided we will not be able to serve request like travel to london or paris
        response = groq.chat.completions.create(model=groq_api_model, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


message array is>>>>>>>>>>>>>>>>>>>>>>>>> [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'Hi'}]
reponse of message array is >>>>>>>>>>>>> ChatCompletion(id='chatcmpl-0cad9fce-5053-410c-883c-9c25353b81c8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you with your travel plans today?', role='assistant', executed_tools=None, function_call=None, reasoning='We need to respond with short, courteous answer, no more than 1 sentence. The user says "Hi". So respond with greeting.', tool_calls=None))], created=1791290175, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint='fp_0adba2bb92', usage=CompletionUsage(completion_tokens=51, prompt_tokens=186, total_tokens=237, completion_time=0.107031844, promp

In [ ]:

# here handle_tool_calls is the same function which is above.
def handle_tool_calls(message):
    responses = []
    print(f"Message in handle_tool_call is >>>>>>>>>> {message}")
    for tool_call in message.tool_calls:
        print(f"And the tool call is >>>>>>>>>>>>>>>>>>>> {tool_call}")
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses


# multiple call like checking route to multimple city
#  we have done improvement in this function by providing tool in the function call under tool call while loop
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(f"message array is>>>>>>>>>>>>>>>>>>>>>>>>>  {messages}")
    response = groq.chat.completions.create(model=groq_api_model, messages=messages, tools=tools)
    print(f"reponse of first message is >>>>>>>>>>>>>  {response}")
    print(f"it has finish reason >>>>>>>>>>>>>>>>>>>>  {response.choices[0].finish_reason}")


    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"message from first message request is>>>>  {message}")
        responses = handle_tool_calls(message)
        print(f"response from handle_tool_call is >>>>>>>  {responses}")
        messages.append(message)
        messages.extend(responses)
        print(f"again request for tool call with messages(previous messages+ response)>>>>>>>>>>  {messages}")
        response = groq.chat.completions.create(model=groq_api_model, messages=messages, tools=tools)
        print(f"again reponse for tool call with above messages(previous messages+ response)>>>>  {response}")
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


message array is>>>>>>>>>>>>>>>>>>>>>>>>>  [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'Hi '}]
reponse of first message is >>>>>>>>>>>>>  ChatCompletion(id='chatcmpl-0a5d5059-03f1-42ef-b31b-9f2b80377cd6', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you with your flight today?', role='assistant', executed_tools=None, function_call=None, reasoning='We need to respond short, courteous, max 1 sentence. User just says "Hi". Should reply greeting.', tool_calls=None))], created=1791290236, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint='fp_90620edd96', usage=CompletionUsage(completion_tokens=45, prompt_tokens=187, total_tokens=232, completion_time=0.093611444, prompt_time=0.007128135, queue_

In [ ]:
import sqlite3


In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

# gets path of current working directory to get to know location of price.db file
print(os.getcwd())


/Users/mikinr1991/Tehcnical_workspace/LLM_Workspace/llm_engineering_by_Mikin/week2


In [54]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [55]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [56]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [57]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [58]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [ ]:
# Have put handle_tool_calls and chat function as as above . here we are just learning how to use database tool
def handle_tool_calls(message):
    responses = []
    print(f"Message in handle_tool_call is >>>>>>>>>> {message}")
    for tool_call in message.tool_calls:
        print(f"And the tool call is >>>>>>>>>>>>>>>>>>>> {tool_call}")
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(f"message array is>>>>>>>>>>>>>>>>>>>>>>>>>  {messages}")
    response = groq.chat.completions.create(model=groq_api_model, messages=messages, tools=tools)
    print(f"reponse of first message is >>>>>>>>>>>>>  {response}")
    print(f"it has finish reason >>>>>>>>>>>>>>>>>>>>  {response.choices[0].finish_reason}")


    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"message from first message request is>>>>  {message}")
        responses = handle_tool_calls(message)
        print(f"response from handle_tool_call is >>>>>>>  {responses}")
        messages.append(message)
        messages.extend(responses)
        print(f"again request for tool call with messages(previous messages+ response)>>>>>>>>>>  {messages}")
        response = groq.chat.completions.create(model=groq_api_model, messages=messages, tools=tools)
        print(f"again reponse for tool call with above messages(previous messages+ response)>>>>  {response}")
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


message array is>>>>>>>>>>>>>>>>>>>>>>>>>  [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'hi'}]
reponse of first message is >>>>>>>>>>>>>  ChatCompletion(id='chatcmpl-7308e66c-f55d-428b-bf52-ca7911375b58', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', role='assistant', executed_tools=None, function_call=None, reasoning='User says "hi". According to instructions, short courteous answer, no more than 1 sentence. So reply "Hello! How can I assist you today?" That\'s one sentence.', tool_calls=None))], created=1791291121, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint='fp_49bfac06f1', usage=CompletionUsage(completion_tokens=55, prompt_tokens=186, total_tokens=241, completion_time

## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>